In [1]:
import os
import sys
import pandas as pd
from datetime import datetime
sys.path.append(os.path.abspath("../../"))
from src.unitelma_processor import UnitelmaProcessor
from src.config_unitelma import CLASSIFIERS

import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
load_dotenv()

ROOT_DIR = os.getenv("ROOT_DIR")

/Volumes/T7/documents/github/dropout-prediction/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
TRAIN_FILE = os.path.join(ROOT_DIR, ".data_unitelma/train_test/train_timeseries.csv")
TEST_FILE = os.path.join(ROOT_DIR, ".data_unitelma/train_test/test_timeseries.csv")
OUTPUT_DIR = os.path.join(ROOT_DIR, "notebooks_unitelma/runs/outputs_run_01")
os.makedirs(OUTPUT_DIR, exist_ok=True)

processor = UnitelmaProcessor(TRAIN_FILE, TEST_FILE, OUTPUT_DIR, False)

[INFO] Device PyTorch -> mps


In [3]:
LAGS = [7, 14]
results = []

for l in LAGS:
    X_train, y_train, X_test, y_test = processor.prepare_ml_data(lag=l, time_aggregation='flatten')

    #X_train, X_test, _ = processor.apply_feature_selection(X_train, y_train, X_test)

    for algo_name, model_config in CLASSIFIERS.items():
        print(f"[TRAIN] {algo_name}...")
        metrics, best_model = processor.evaluate_ml_model(model_config, X_train, y_train, X_test, y_test)
        
        metrics['lag'] = l
        metrics['algorithm'] = algo_name
        metrics.pop('best_params', None)
        results.append(metrics)



if results:
    df_results = pd.DataFrame(results)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    csv_path = os.path.join(OUTPUT_DIR, f"metrics_ml_{timestamp}.csv")
    
    df_results.to_csv(csv_path, index=False)
    print(f"\n[SUCCESS] Complete -> {csv_path}")
    
    # Mostriamo la tabella riassuntiva a video nel notebook
    display(df_results)

[TRAIN] LogisticRegression...
[TRAIN] RandomForest...
[TRAIN] LogisticRegression...
[TRAIN] RandomForest...

[SUCCESS] Complete -> /Volumes/T7/documents/github/dropout-prediction/notebooks_unitelma/runs/outputs_run_01/metrics_ml_20260531_215750.csv


,accuracy,precision,recall,f1,roc_auc,pr_auc,lag,algorithm
0,0.7122,0.6980,0.7122,0.7047,0.5710,0.7086,7,LogisticRegression
1,0.6831,0.7835,0.6831,0.7118,0.7310,0.7920,7,RandomForest
2,0.7277,0.6934,0.7277,0.7081,0.5417,0.6990,14,LogisticRegression
3,0.7064,0.7560,0.7064,0.7251,0.7274,0.7945,14,RandomForest
